# SemiRestore — Semiconductor Inspection Research Notebook + CLI

This is the **research-code deliverable**: Python notebook, reproducible benchmark, experimental feature ablations, dataset-factory tooling, regression tests, and command-line inference. It intentionally does **not** create a UI, hosted web app, or persistent API.

The first setup cell tries to clone this GitHub repository, or falls back to uploading the project ZIP, then checks/installs dependencies and regenerates only missing fixed-seed synthetic PNGs. **Runtime → Run all** executes a live sample inference, builds the per-chip coordinate/inspection tables and annotated images, loads the saved 480-prediction benchmark and ablation evidence, checks the optional dataset-factory interfaces, and runs the regression suite. The full benchmark recomputation and experimental ranker training are explicit opt-ins via `RUN_FULL_HOLDOUT_BENCHMARK = True` and `RUN_ML_STRUCTURE_ABLATION = True`; they are separated so a normal Run all finishes more reliably on Colab CPU. The benchmark can be run in three sequential 20-pair chunks from the runbook.

**Evidence boundary:** the included DRAM/FinFET pairs and coordinates are synthetic. The center-prior hybrid is strong on centered synthetic pairs and materially weaker under translation. The gradient/FFT/Random Forest verifier is experimental and is not used by the evaluator-facing default CLI because prior validation did not show reliable improvement. Unclassified anomaly candidates are for review only; there is no validated defect classifier. Nothing here claims production or real-wafer accuracy.

In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, json, csv, shutil, importlib.util
import numpy as np

try:
    import google.colab as _colab_probe  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def is_project_root(path: Path) -> bool:
    """Accept a usable code root even when optional saved benchmark files are absent."""
    required = ('src/nova_inference.py', 'src/generator.py', 'requirements.txt',
                'scripts/prepare_synthetic_dataset.py')
    return all((path / rel).is_file() for rel in required)


def locate_project_root():
    """Find a project extracted flat or under an extra enclosing directory."""
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path('/content/SemiRestore'), Path('/content/SemiRestore_ZIP'),
                  Path('/content/NOVA-Loc_Integrated_Project'),
                  Path('/content/NOVA_Loc_Colab_Inspection_Starter')]
    if Path('/content').exists():
        candidates.extend(p.parent.parent for p in Path('/content').rglob('src/nova_inference.py'))
    seen = set()
    for candidate in candidates:
        try:
            root = candidate.resolve()
        except OSError:
            continue
        if root in seen:
            continue
        seen.add(root)
        if is_project_root(root):
            return root
    return None

PROJECT_ROOT = locate_project_root()
if PROJECT_ROOT is None and IN_COLAB:
    # Preferred route: clone the now-public repository (avoids broken ChatGPT downloads).
    repo_target = Path('/content/SemiRestore')
    repo_url = 'https://github.com/RAJARYANSINGH0059/SemiRestore.git'
    if repo_target.exists() and not is_project_root(repo_target):
        shutil.rmtree(repo_target, ignore_errors=True)  # remove an earlier partial extraction
    if not is_project_root(repo_target):
        clone = subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo_target)],
                               capture_output=True, text=True)
        if clone.returncode != 0:
            print('GitHub clone failed; falling back to project ZIP upload.')
            print(clone.stderr[-1200:])
    PROJECT_ROOT = locate_project_root()

if PROJECT_ROOT is None and IN_COLAB:
    from google.colab import files
    print('Upload the SemiRestore project ZIP (flat compact ZIP or full project ZIP).')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No ZIP uploaded. Please choose the SemiRestore project ZIP.')
    dest = Path('/content/SemiRestore_ZIP')
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(Path('/content') / zip_names[0]) as archive:
        for member in archive.infolist():
            target = (dest / member.filename).resolve()
            if target != dest.resolve() and dest.resolve() not in target.parents:
                raise ValueError(f'Unsafe path in uploaded ZIP: {member.filename}')
        archive.extractall(dest)
    PROJECT_ROOT = locate_project_root()
    if PROJECT_ROOT is None:
        found = [str(p.parent.parent) for p in dest.rglob('src/nova_inference.py')]
        raise FileNotFoundError(
            'ZIP extracted, but SemiRestore root was not detected. Expected src/nova_inference.py, '
            'src/generator.py, requirements.txt, and scripts/prepare_synthetic_dataset.py. '
            f'Candidate roots: {found[:8]}'
        )

if PROJECT_ROOT is None:
    raise FileNotFoundError('SemiRestore root not found. Open this notebook from the repository or upload its ZIP.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Install only when imports are missing; avoid unnecessary pip/network access.
requirements_path = PROJECT_ROOT / 'requirements.txt'
required_modules = ['numpy', 'PIL', 'cv2', 'pytest', 'pandas', 'matplotlib', 'yaml', 'sklearn', 'joblib']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    if IN_COLAB and requirements_path.is_file():
        print('Installing missing dependencies:', missing_modules)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
    else:
        raise ImportError(f'Missing packages {missing_modules}; install requirements.txt first.')
else:
    print('All notebook/test dependencies are importable; skipped pip install.')

# A compact ZIP may omit reference/search PNGs and some optional result artifacts.
# Generate ground truth/images only when needed; never overwrite existing images.
needs_prepare = False
for arch in ('DRAM', 'FinFET'):
    gt_path = PROJECT_ROOT / 'results' / 'baseline' / arch / 'ground_truth.csv'
    if not gt_path.is_file():
        needs_prepare = True
        continue
    with gt_path.open(newline='', encoding='utf-8') as fh:
        gt_rows = list(csv.DictReader(fh))
    if len(gt_rows) != 30:
        raise ValueError(f'Expected 30 rows in {gt_path}, found {len(gt_rows)}')
    for row in gt_rows:
        for key in ('reference', 'search'):
            item = Path(row[key])
            if not item.is_absolute():
                item = PROJECT_ROOT / item
            if not item.is_file():
                needs_prepare = True
                break
        if needs_prepare:
            break
if needs_prepare:
    print('Ground truth or sample PNGs are missing; running fixed-seed dataset preparation.')
    subprocess.check_call([sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_synthetic_dataset.py')], cwd=PROJECT_ROOT)
else:
    print('Both ground-truth CSVs and all 60 reference/search pairs are present.')

# If a minimalist code-only archive omitted saved benchmark results, build them in 3 chunks.
metrics_file = PROJECT_ROOT / 'results' / 'integrated_benchmark' / 'integrated_metrics_summary.json'
ledger_file = PROJECT_ROOT / 'results' / 'integrated_benchmark' / 'integrated_per_sample_results.csv'
if not metrics_file.is_file() or not ledger_file.is_file():
    print('Saved 480-row benchmark ledger is absent; computing 3 chunks. This may take several minutes on CPU.')
    for i, start_index in enumerate((0, 20, 40), start=1):
        part_dir = PROJECT_ROOT / 'results' / 'benchmark_parts' / f'part{i}'
        subprocess.check_call([sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_integrated_benchmark.py'),
                               '--start-index', str(start_index), '--limit', '20', '--out-dir', str(part_dir)], cwd=PROJECT_ROOT)
    subprocess.check_call([sys.executable, str(PROJECT_ROOT / 'scripts' / 'merge_benchmark_parts.py')], cwd=PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('Colab runtime:', IN_COLAB)


## 0. Configure which research runs execute

By default, `Run all` executes fresh CLI inference, creates the inspection tables/annotated images, loads the saved benchmark ledger and prior ablation evidence, checks optional dataset-factory command interfaces, and runs tests. The full 480-prediction benchmark recomputation and fresh-seed ranker experiment are explicit opt-ins because full inference can take several minutes or longer on CPU. Set `RUN_FULL_HOLDOUT_BENCHMARK = True` to recompute the benchmark, `RUN_ML_STRUCTURE_ABLATION = True` to rerun the small experimental ranker validation, and `RUN_UNIFIED_FACTORY = True` only after placing licensed/source images under `SOURCE_ROOT`.

In [ ]:
# Execution switches: the default Run all builds the inspection report as well as running research tests.
RUN_FULL_HOLDOUT_BENCHMARK = False  # set True to recompute all 480 predictions (may take several minutes)
RUN_ML_STRUCTURE_ABLATION = False   # set True for separate-seed experimental ranker validation
RUN_INSPECTION_REPORT = True        # per-pair visualization + per-chip/site/anomaly coordinate tables
RUN_UNIFIED_FACTORY = False         # requires user-supplied external images
SHOW_BENCHMARK_PLOTS = False        # the main output is tables + simple annotated image, not lots of graphs
SHOW_INLINE_OVERLAY = False       # set True to embed the large JPEG inside Colab; the file is always saved
SOURCE_ROOT = Path('/content/drive/MyDrive/NOVA_Loc_Data/sources')

# Leave both chip dimensions None if each search image is one inspection field/chip.
# Only set these to measured dimensions when a search image contains a regular grid of multiple chips/dies.
CHIP_WIDTH_PX = None
CHIP_HEIGHT_PX = None
ANOMALY_THRESHOLD_SIGMA = 2.5
MAX_ANOMALY_CANDIDATES_PER_IMAGE = 20  # top-K shortlist; all candidates are always saved to anomaly_candidates_all.csv
DEFECT_ANNOTATIONS_CSV = None  # optionally set to 'data/defect_annotations.csv' after filling data/defect_annotations_template.csv
ANNOTATE_TOP_ANOMALIES = 12

print('RUN_FULL_HOLDOUT_BENCHMARK =', RUN_FULL_HOLDOUT_BENCHMARK)
print('RUN_ML_STRUCTURE_ABLATION =', RUN_ML_STRUCTURE_ABLATION)
print('RUN_INSPECTION_REPORT =', RUN_INSPECTION_REPORT)
print('RUN_UNIFIED_FACTORY =', RUN_UNIFIED_FACTORY)
print('SHOW_BENCHMARK_PLOTS =', SHOW_BENCHMARK_PLOTS)
print('SHOW_INLINE_OVERLAY =', SHOW_INLINE_OVERLAY)
print('This notebook is Python + CLI only; no UI/API/server is started.')


## 1. Inspect the supplied dataset and source contract
The CSVs in the ZIP are generated ground-truth pairs. We verify counts and list the inference implementation actually loaded.

In [ ]:
from src.nova_inference import locate
from src.baseline import baseline_fixed, baseline_multiscale, baseline_center_aware
import cv2

rows = []
for arch in ('DRAM', 'FinFET'):
    gt_path = PROJECT_ROOT / 'results' / 'baseline' / arch / 'ground_truth.csv'
    with gt_path.open(newline='') as f:
        arch_rows = list(csv.DictReader(f))
    rows.extend([{**r, 'architecture': arch} for r in arch_rows])
    print(f'{arch}: {len(arch_rows)} pairs; ground-truth CSV: {gt_path.relative_to(PROJECT_ROOT)}')
print('Total source pairs:', len(rows))
print('Inference module:', Path(__import__('src.nova_inference', fromlist=['x']).__file__).relative_to(PROJECT_ROOT))


## 2. Run one real bundled source pair (synthetic image)
This executes the same evaluator-facing inference module used by `src/infer.py`. Ground truth is taken from its CSV; no coordinate is hardcoded into the inference function.

In [ ]:
import subprocess, pandas as pd
from src.nova_inference import locate
from src.baseline import baseline_fixed, baseline_multiscale, baseline_center_aware

sample_gt = rows[0]
ref_path = PROJECT_ROOT / sample_gt['reference']
search_path = PROJECT_ROOT / sample_gt['search']
gt_x, gt_y = float(sample_gt['x']), float(sample_gt['y'])
result = locate(ref_path, search_path)
err = float(np.hypot(result.x - gt_x, result.y - gt_y))
print('Architecture:', sample_gt['architecture'])
print('Reference:', ref_path)
print('Search:', search_path)
print(f'Ground truth center: ({gt_x:.3f}, {gt_y:.3f})')
print(f'NOVA hybrid prediction: ({result.x:.3f}, {result.y:.3f}); error={err:.3f}px')
print('Detailed diagnostics:', json.dumps(result.to_dict(), indent=2))

# Execute the evaluator-facing CLI on this same real file pair. Its stdout
# contract is exactly two numeric coordinates; diagnostics stay in the API.
cli = subprocess.run([sys.executable, str(PROJECT_ROOT/'src'/'infer.py'), str(ref_path), str(search_path)],
                     cwd=PROJECT_ROOT, capture_output=True, text=True, check=True)
coords = cli.stdout.strip().split()
assert len(coords) == 2 and all(np.isfinite(float(v)) for v in coords), cli.stdout
print('CLI stdout (exact x y contract):', cli.stdout.strip())

baseline_rows = []
score, xy, *_ = baseline_fixed(ref_path, search_path)
baseline_rows.append({'Method':'B1 fixed-scale NCC','x':xy[0],'y':xy[1],'score':score})
score, xy, *_ = baseline_multiscale(ref_path, search_path)
baseline_rows.append({'Method':'B2 multiscale NCC','x':xy[0],'y':xy[1],'score':score})
score, xy, *_ = baseline_center_aware(ref_path, search_path)
baseline_rows.append({'Method':'B3 center-aware NCC','x':xy[0],'y':xy[1],'score':score})
baseline_rows.append({'Method':'NOVA hybrid ROI + ambiguity gate','x':result.x,'y':result.y,'score':result.score})
for row in baseline_rows:
    row['error_px'] = float(np.hypot(row['x']-gt_x, row['y']-gt_y))
display(pd.DataFrame(baseline_rows))

## 3. Benchmark design — centered and translated holdout
The benchmark evaluates 60 distinct held-out images × four methods in each condition. The translation condition uses known deterministic offsets of up to ±50 px and updates ground truth by the same offsets. This is a synthetic drift stress test, not physical wafer-stage drift.

## 3.1 Execute the complete held-out benchmark (live)

This optional stage reruns all 60 held-out synthetic pairs in centered and deterministic ±50 px translated conditions, comparing B1/B2/B3 and the hybrid. It executes in three sequential chunks and validates the freshly merged 480-row ledger. The default Run all reads the committed ledger for reliability; set `RUN_FULL_HOLDOUT_BENCHMARK = True` in the configuration cell to recompute all predictions on the images generated by your current runtime. On CPU, the full job may take several minutes or longer.

In [ ]:
import subprocess, sys, time

if RUN_FULL_HOLDOUT_BENCHMARK:
    print('Running full held-out benchmark: 3 sequential chunks, 480 predictions total.')
    chunks = [
        (0, 20, 'part1'),
        (20, 20, 'part2'),
        (40, 20, 'part3'),
    ]
    for start, limit, name in chunks:
        cmd = [sys.executable, 'scripts/run_integrated_benchmark.py',
               '--start-index', str(start), '--limit', str(limit),
               '--out-dir', f'results/benchmark_parts/{name}']
        print('\n$', ' '.join(cmd), flush=True)
        proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            # Stream only useful progress lines; preserve the full merged CSV/JSON on disk.
            if any(token in line for token in ('Loaded ', 'Holdout pair ', 'Wrote:')):
                print(line.rstrip(), flush=True)
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f'Benchmark chunk {name} failed with exit code {rc}')
    merge_cmd = [sys.executable, 'scripts/merge_benchmark_parts.py']
    print('\nMerging and validating the 480 result records...')
    subprocess.run(merge_cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('Skipping live benchmark because RUN_FULL_HOLDOUT_BENCHMARK=False; using saved benchmark artifacts.')

In [ ]:
import pandas as pd
from IPython.display import display
summary = json.loads((PROJECT_ROOT / 'results/integrated_benchmark/integrated_metrics_summary.json').read_text())
method_names = {
    'B1_fixed_ncc': 'B1 fixed 10x NCC',
    'B2_multiscale_ncc': 'B2 7-scale NCC',
    'B3_center_aware_ncc': 'B3 original center-aware NCC',
    'NOVA_hybrid_roi_gate': 'NOVA bounded ROI + late center gate',
}

def make_summary_table(split, arch='ALL'):
    out=[]
    for key, method in method_names.items():
        d=summary[split][key][arch]
        out.append({'Method':method, 'n':d['n'], '≤1 px':f"{d['accuracy_1px']:.1%}", '≤2 px':f"{d['accuracy_2px']:.1%}", '≤5 px':f"{d['accuracy_5px']:.1%}", '≤10 px':f"{d['accuracy_10px']:.1%}", 'Mean error (px)':round(d['mean_error_px'],2), 'P95 error (px)':round(d['p95_error_px'],2), 'Mean latency (ms)':round(d['mean_latency_ms'],1)})
    return pd.DataFrame(out)

print('CENTERED HOLDOUT — target is generated at the center by the source data generator')
display(make_summary_table('holdout_centered'))
print('TRANSLATED HOLDOUT — exact target shifted by known ±50 px transformations')
display(make_summary_table('holdout_translated'))


In [ ]:
print('Architecture breakdown: centered')
display(pd.concat([make_summary_table('holdout_centered', 'DRAM').assign(Architecture='DRAM'), make_summary_table('holdout_centered','FinFET').assign(Architecture='FinFET')], ignore_index=True)[['Architecture','Method','n','≤1 px','≤5 px','Mean error (px)','P95 error (px)','Mean latency (ms)']])
print('Architecture breakdown: translated')
display(pd.concat([make_summary_table('holdout_translated', 'DRAM').assign(Architecture='DRAM'), make_summary_table('holdout_translated','FinFET').assign(Architecture='FinFET')], ignore_index=True)[['Architecture','Method','n','≤1 px','≤5 px','Mean error (px)','P95 error (px)','Mean latency (ms)']])


## 6. Automated regression tests
These tests check inference repeatability, the evaluator CLI contract, invalid/missing input handling, and the merged holdout record count. They do not prove real-world generalization.

In [ ]:
test_run = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=180)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError(f'pytest failed with exit code {test_run.returncode}')


## 3.2 Per-chip inspection output — coordinates, anomaly candidates, and annotated images

This is the main human-readable inspection output. It saves one annotated reference/search image per sample (red × = predicted localization, green + = known ground truth when available, orange squares = heuristic anomaly candidates), plus three CSV tables:

- `results/inspection_report/inspection_sites.csv`: one row per reference/search pair with image-global and chip-local coordinates, score, scale, ambiguity-gate details, hash and localization error when ground truth exists.
- `results/inspection_report/chip_summary.csv`: one row per chip/tile, target coordinates local to that tile, plus counts of unclassified anomaly candidates.
- `results/inspection_report/anomaly_candidates.csv`: top-K shortlist (20 per image by default) for quick review.
- `results/inspection_report/anomaly_candidates_all.csv`: every thresholded heuristic candidate with global/local coordinates and bounding box; this is the complete candidate table, not a list of confirmed defects.
- `results/inspection_report/reviewed_defect_annotations.csv`: expert-reviewed defect labels and global/local coordinates, if `DEFECT_ANNOTATIONS_CSV` is supplied; otherwise it is empty.

**Coordinate rule:** image-global coordinates are pixel-center positions in the search image with origin at the top-left. Chip-local coordinates subtract the top-left origin of the configured chip tile. Since the supplied synthetic data has no physical chip/die boundary map, the default treats each search image as one inspection field. If your real search image contains a regular grid of multiple chips/dies, set `CHIP_WIDTH_PX` and `CHIP_HEIGHT_PX` in the configuration cell to measured sizes; the notebook will create tile IDs and local coordinates and draw tile boundaries.

**Defect-label limitation:** the current SemiRestore / NOVA-Loc pipeline is a template-localization method. It does not have supervised defect labels or a validated defect-type classifier. The orange squares and candidate CSVs are unclassified local-contrast outliers for review, not confirmed defects (they can include ordinary periodic features/noise). `anomaly_candidates_all.csv` always contains all thresholded candidates. `MAX_ANOMALY_CANDIDATES_PER_IMAGE` only controls the shortlist/annotation density. If you have reviewer labels, copy `data/defect_annotations_template.csv` to `data/defect_annotations.csv`, fill in the actual labels and pixel coordinates, then set `DEFECT_ANNOTATIONS_CSV = 'data/defect_annotations.csv'` in the configuration cell. Do not set `confirmed=True` unless confirmed by a qualified reviewer/ground truth.


In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image as NotebookImage

if RUN_INSPECTION_REPORT:
    report_dir = PROJECT_ROOT / 'results' / 'inspection_report'
    cmd = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'build_inspection_report.py'),
        '--root', str(PROJECT_ROOT),
        '--output-dir', str(report_dir),
        '--split', 'holdout_centered',
        '--method', 'NOVA_hybrid_roi_gate',
        '--threshold-sigma', str(ANOMALY_THRESHOLD_SIGMA),
        '--min-area', '6',
        '--max-area', '250',
        '--max-candidates-per-image', str(MAX_ANOMALY_CANDIDATES_PER_IMAGE),
        '--annotate-anomalies', str(ANNOTATE_TOP_ANOMALIES),
    ]
    if CHIP_WIDTH_PX is not None:
        cmd += ['--chip-width-px', str(CHIP_WIDTH_PX)]
    if CHIP_HEIGHT_PX is not None:
        cmd += ['--chip-height-px', str(CHIP_HEIGHT_PX)]
    if DEFECT_ANNOTATIONS_CSV:
        annotation_path = Path(DEFECT_ANNOTATIONS_CSV)
        if not annotation_path.is_absolute():
            annotation_path = PROJECT_ROOT / annotation_path
        cmd += ['--defect-annotations', str(annotation_path)]

    print('Building inspection report in a separate, bounded CLI process…', flush=True)
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=180)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
        raise RuntimeError(f'Inspection report process failed (exit={proc.returncode})')
    # The CLI prints progress lines before its JSON payload, so read the canonical report file.
    summary_json = report_dir / 'inspection_report.json'
    if not summary_json.is_file():
        print(proc.stdout[-4000:])
        raise FileNotFoundError(f'Inspection summary was not written: {summary_json}')
    report_summary = json.loads(summary_json.read_text(encoding='utf-8'))

    print('INSPECTION REPORT SUMMARY')
    summary_cols = {
        'Search images/sites': report_summary.get('sample_count'),
        'Chip/tile rows': report_summary.get('chip_tile_count'),
        'All unconfirmed heuristic candidates': report_summary.get('anomaly_candidate_count'),
        'Top-K shortlist candidates': report_summary.get('shortlist_candidate_count'),
        'Expert-reviewed defect labels loaded': report_summary.get('expert_defect_annotations_loaded'),
        'Confirmed expert labels': report_summary.get('confirmed_expert_defect_annotations'),
        'Ground-truth rows': report_summary.get('ground_truth_rows'),
        'Rows without ground truth': report_summary.get('ground_truth_unavailable_rows'),
        'Output directory': str(report_dir),
    }
    display(pd.DataFrame([summary_cols]))

    # Keep notebook rendering intentionally compact; complete registers remain on disk.
    site_df = pd.read_csv(report_dir / 'inspection_sites.csv')
    site_cols = [c for c in ['sample_id','architecture','chip_id','pred_x_image_px','pred_y_image_px',
                 'pred_x_chip_local_px','pred_y_chip_local_px','ground_truth_x_image_px',
                 'ground_truth_y_image_px','localization_error_px','match_score_ncc',
                 'chosen_scale_ratio','anomaly_candidate_count','defect_assessment_status'] if c in site_df]
    print('SITE / LOCALIZATION TABLE — first 8 rows; full table in CSV')
    display(site_df[site_cols].head(8))

    chip_df = pd.read_csv(report_dir / 'chip_summary.csv')
    chip_cols = [c for c in ['sample_id','architecture','chip_id','chip_row','chip_col','chip_origin_x_px',
                 'chip_origin_y_px','pred_x_chip_local_px','pred_y_chip_local_px',
                 'anomaly_candidate_count','bright_candidate_count','dark_candidate_count',
                 'mixed_candidate_count','confirmed_defect_count'] if c in chip_df]
    print('PER-CHIP / PER-TILE SUMMARY — first 8 rows; full table in CSV')
    display(chip_df[chip_cols].head(8))

    anomaly_df = pd.read_csv(report_dir / 'anomaly_candidates.csv')
    anomaly_cols = [c for c in ['sample_id','architecture','chip_id','candidate_id','x_image_px','y_image_px',
                    'x_chip_local_px','y_chip_local_px','bbox_width_px','bbox_height_px',
                    'candidate_kind','observation','severity','defect_type','confirmation_status'] if c in anomaly_df]
    print('UNCLASSIFIED ANOMALY CANDIDATES — first 12 shortlisted rows')
    if len(anomaly_df):
        sort_cols = [c for c in ['severity','robust_anomaly_score'] if c in anomaly_df]
        if len(sort_cols) == 2:
            anomaly_df = anomaly_df.sort_values(['severity','robust_anomaly_score'], ascending=[True,False])
        display(anomaly_df[anomaly_cols].head(12))
    else:
        print('No candidates exceeded the heuristic thresholds; this does not establish that images are defect-free.')

    all_anomaly_path = report_dir / 'anomaly_candidates_all.csv'
    # Count without loading the entire multi-megabyte file into the notebook display layer.
    with all_anomaly_path.open('r', encoding='utf-8', errors='replace') as fh:
        all_candidate_count = max(0, sum(1 for _ in fh) - 1)
    print(f'COMPLETE CANDIDATE REGISTER: {all_candidate_count} thresholded heuristic candidates.')
    print('Complete CSV:', all_anomaly_path)

    reviewed_path = report_dir / 'reviewed_defect_annotations.csv'
    reviewed_df = pd.read_csv(reviewed_path)
    if len(reviewed_df):
        print('EXPERT-REVIEWED DEFECT LABELS')
        display(reviewed_df.head(12))
    else:
        print('No expert defect labels supplied. Candidates remain UNCLASSIFIED/UNCONFIRMED; see data/defect_annotations_template.csv.')

    overlay = report_dir / 'sample_inspection_overlay.jpg'
    print('Annotated prediction/ground-truth example:', overlay)
    if SHOW_INLINE_OVERLAY and overlay.is_file():
        display(NotebookImage(filename=str(overlay)))
    else:
        print('Inline JPEG preview is off for lighter Run all. Open the saved file or set SHOW_INLINE_OVERLAY=True.')
    print('Per-image annotated outputs:', report_dir / 'visuals')
    print('Report files:', report_dir)
else:
    print('Skipped inspection reporting by configuration.')


## 4. What happened to the ML / structural ablations?
The gradient/FFT/RF candidate ranker was tested on separately generated validation pairs. The prior measured result was not reliable enough to promote it into default inference. It remains experimental and is not used in `src/infer.py`.

## 5. Run the experimental gradient / FFT / Random Forest feature pipeline

This opt-in-by-default diagnostic generates separate training and validation examples from different seeds, trains the candidate-level Random Forest, and evaluates structural/ML ranking variants on the held-out generated validation CSV. It does **not** touch the 60-pair holdout, and its model is not enabled in `src/infer.py`. These synthetic ablations are for testing the feature code and recording evidence; do not treat them as real-wafer metrics.

In [ ]:
if RUN_ML_STRUCTURE_ABLATION:
    import csv, shutil, joblib
    from src.generator import generate
    from src.verification import train_ranker, score_dataset
    runtime_root = PROJECT_ROOT / 'work_results' / 'notebook_ml_ablation'
    if runtime_root.exists():
        shutil.rmtree(runtime_root)
    train_paths, val_paths = [], []
    for arch in ('DRAM', 'FinFET'):
        train_paths.append(generate(arch, 6, runtime_root/'train'/arch, seed=770100 + (0 if arch == 'DRAM' else 10000)))
        val_paths.append(generate(arch, 3, runtime_root/'validation'/arch, seed=880200 + (0 if arch == 'DRAM' else 10000)))

    # Combine validation records for one held-out evaluation call.
    val_csv = runtime_root/'validation'/'combined_validation.csv'
    combined=[]
    for gt in val_paths:
        with gt.open(newline='') as f:
            combined.extend(list(csv.DictReader(f)))
    fields=list(combined[0].keys())
    with val_csv.open('w', newline='') as f:
        writer=csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(combined)

    model_path = runtime_root/'models'/'candidate_ranker.joblib'
    train_info = train_ranker(train_paths, model_path, seed=20261027)
    print('Experimental ranker training diagnostics:')
    display(pd.DataFrame([{'Training pairs':train_info['training_pairs'],
                           'Training candidates':train_info['training_candidates'],
                           'Positive candidate rate':train_info['positive_candidate_rate'],
                           'Candidate recall @ 5 px (training)':train_info['candidate_recall_at_100'],
                           'Model path':train_info['model_path']}]))
    print('Feature importances (diagnostic only):')
    display(pd.DataFrame([{'Feature':k,'Importance':v} for k,v in train_info['feature_importances'].items()]).sort_values('Importance', ascending=False))

    model = joblib.load(model_path)
    val_detail, val_summary = score_dataset(val_csv, model, runtime_root/'validation_scored', 'seed_heldout_validation')
    val_method_rows=[]
    for method, metrics in val_summary['methods'].items():
        val_method_rows.append({'Method':method,'n':val_summary['n'],
                                'Candidate recall ≤5 px':val_summary['candidate_recall_at_5px'],
                                'Accuracy ≤5 px':metrics['accuracy_5px'],
                                'Mean error (px)':metrics['mean_error_px'],
                                'P95 error (px)':metrics['p95_error_px']})
    print('Fresh ML/structure validation metrics; separate from the 60-pair holdout:')
    display(pd.DataFrame(val_method_rows))
    print('Saved validation details:', runtime_root/'validation_scored'/'seed_heldout_validation_per_sample.csv')
else:
    print('Skipped ML/structure ablation by configuration.')

In [ ]:
ablation_dir = PROJECT_ROOT / 'results' / 'ablation'
val_rows=[]
for filename, split_name in [('validation_summary.json','Centered validation'), ('validation_shifted_summary.json','Translated validation')]:
    p=ablation_dir / filename
    if not p.is_file():
        continue
    d=json.loads(p.read_text())
    for method, m in d.get('methods',{}).items():
        val_rows.append({'Split':split_name, 'Method':method, 'n':d.get('n'), 'Candidate recall ≤5 px':d.get('candidate_recall_at_5px'), 'Accuracy ≤5 px':m.get('accuracy_5px'), 'Mean error (px)':m.get('mean_error_px')})
val_df=pd.DataFrame(val_rows)
if len(val_df):
    val_df['Accuracy ≤5 px']=val_df['Accuracy ≤5 px'].map(lambda x: f'{x:.1%}' if x is not None else 'n/a')
    val_df['Candidate recall ≤5 px']=val_df['Candidate recall ≤5 px'].map(lambda x: f'{x:.1%}' if x is not None else 'n/a')
    val_df['Mean error (px)']=val_df['Mean error (px)'].map(lambda x: round(x,2) if x is not None else None)
    display(val_df)
else:
    print('Validation ablation result files not found.')


In [ ]:
if SHOW_BENCHMARK_PLOTS:
    import matplotlib.pyplot as plt
    plot_summary = []
    for split, label in [('holdout_centered','Centered'), ('holdout_translated','Translated ±50 px')]:
        for key, method in method_names.items():
            plot_summary.append({'Condition':label, 'Method':method,
                                 'Accuracy ≤5 px':summary[split][key]['ALL']['accuracy_5px']*100})
    plot_df = pd.DataFrame(plot_summary)
    for label in ['Centered','Translated ±50 px']:
        sub = plot_df[plot_df['Condition']==label]
        plt.figure(figsize=(8,3.5))
        plt.bar(sub['Method'], sub['Accuracy ≤5 px'])
        plt.ylabel('Accuracy within 5 px (%)')
        plt.title(f'Synthetic holdout — {label}')
        plt.ylim(0,100)
        plt.xticks(rotation=15, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('Benchmark graphs are disabled by default. Use the numeric metric tables and per-image overlays above.')


## 7. Optional unified dataset factory (requires user-provided source images)

The unified dataset factory is optional and is not executed during the default Run all because third-party source images are not bundled. When licensed source images are supplied, set `RUN_UNIFIED_FACTORY = True` to build a manifest, generate pairs, and create source-level splits. Set `RUN_UNIFIED_FACTORY = True` in the configuration cell after placing source images below `SOURCE_ROOT`.

In [ ]:
from pathlib import Path
import subprocess, sys

factory_scripts = [
    PROJECT_ROOT/'external/unified_dataset_factory/scripts/build_manifest.py',
    PROJECT_ROOT/'external/unified_dataset_factory/scripts/generate_unified_pairs.py',
    PROJECT_ROOT/'external/unified_dataset_factory/scripts/make_splits.py',
]
if RUN_UNIFIED_FACTORY:
    # Only exercise the external-data factory after the user supplies licensed source images.
    for script in factory_scripts:
        p = subprocess.run([sys.executable, str(script), '--help'], cwd=PROJECT_ROOT,
                           capture_output=True, text=True, timeout=30)
        assert p.returncode == 0, f'Factory interface check failed: {script.name}\n{p.stderr}'
        print(f'OK: {script.relative_to(PROJECT_ROOT)} CLI interface')
    if not SOURCE_ROOT.is_dir():
        raise FileNotFoundError(f'SOURCE_ROOT does not exist: {SOURCE_ROOT}. Mount Drive and add licensed source images first.')
    generated_root = PROJECT_ROOT/'work_results'/'unified_factory_run'
    manifest = PROJECT_ROOT/'data'/'source_manifest.csv'
    subprocess.run([sys.executable, str(factory_scripts[0]), '--root', str(SOURCE_ROOT), '--out', str(manifest), '--hash'], cwd=PROJECT_ROOT, check=True, timeout=120)
    subprocess.run([sys.executable, str(factory_scripts[1]), '--source-root', str(SOURCE_ROOT), '--out', str(generated_root), '--pairs', '200', '--reference-size', '64', '--search-size', '1000', '--scale', '10', '--seed', '42', '--architecture', 'mixed'], cwd=PROJECT_ROOT, check=True, timeout=600)
    metadata = generated_root/'metadata.jsonl'
    subprocess.run([sys.executable, str(factory_scripts[2]), '--metadata', str(metadata), '--out', str(generated_root/'splits.json')], cwd=PROJECT_ROOT, check=True, timeout=60)
    print('Dataset factory output:', generated_root)
else:
    print('Unified dataset factory skipped by default: it requires external, licensed source images.')
    print('Available tools:', ', '.join(p.name for p in factory_scripts))


## 8. Interpretation, limitations, and next real-data step

- The hybrid reaches 100% within 5 px on the centered synthetic holdout because the source generator deliberately places the target at the center and the method uses a bounded center prior. This is not a fair proxy for arbitrary drift.
- Under known ±50 px translations, hybrid accuracy drops to 51.7% within 5 px. The bounded ROI + center ambiguity gate is not yet a robust drift solution.
- The RF candidate ranker and gradient/periodicity variants did not consistently beat the baseline on held-out validation. They remain experimental/off by default rather than being promoted because they are labelled AI.
- The inference API has only a cropped reference and full search image; no neighboring reference-context image is supplied, so a true context-ring similarity is not calculated.
- These images are generated, synthetic SEM-inspired patterns. Real wafer SEM data, expert localization annotations, stage calibration, and pixel-to-nanometre calibration remain required for industrial claims.

### Re-run the complete benchmark

From a terminal in the project root, run each 20-pair chunk and merge it:

```bash
python scripts/run_integrated_benchmark.py --start-index 0 --limit 20 --out-dir results/benchmark_parts/part1
python scripts/run_integrated_benchmark.py --start-index 20 --limit 20 --out-dir results/benchmark_parts/part2
python scripts/run_integrated_benchmark.py --start-index 40 --limit 20 --out-dir results/benchmark_parts/part3
python scripts/merge_benchmark_parts.py
```
